# Reproducing Scala, Kobak et al. (2021) from dandiset 000035

Paper: Scala F, Kobak D, Bernabucci M, et al. (2021). Phenotypic variation of transcriptomic
cell types in mouse motor cortex. *Nature* 598:144-150 (published online in 2020).
[doi:10.1038/s41586-020-2907-3](https://doi.org/10.1038/s41586-020-2907-3). The full text was
used (open access, CC BY, read from Europe PMC, PMC8113357).

Data: [DANDI:000035](https://dandiarchive.org/dandiset/000035/0.211014.0808), version
0.211014.0808, streamed from the archive.

Why this version: The paper links the dandiset without a version. 0.211014.0808 is the only
published version. The draft holds the same 185 files (same paths and sizes, checked below) and
differs only in dandiset-level metadata, so nothing the analysis uses differs between them.

## What can and cannot be tested

The paper's main data set (1,329 neurons recorded at room temperature, with sequencing and
morphologies) is in dandiset 000008 and other archives. Dandiset 000035 holds only the follow-up
experiments at physiological temperature: 185 neurons, mostly layer 5 Sst and excitatory cells,
one NWB file per neuron with a ladder of current steps and the neuron's t-type, layer and soma
depth. No gene counts or morphologies are in it.

| Claim | What | Here |
|---|---|---|
| `fig1-ttype-coverage`, `fig2-rrr-prediction`, `ed5-rrr-ion-channels`, `fig3d-families-separate`, `fig3d-morphology-combined`, `fig4c-distance-correlation`, `fig4d-it-depth`, `fig5a-ttypes-between-families`, `fig5b-ttype-variability` | Findings on the main room-temperature data set | Blocked: not in this dandiset |
| `ed6-cohort` | Counts of cells, mice and t-types at 34 degrees | Tested |
| `ed6-stimulus-protocol` | 600 ms steps from -200 pA in 20 pA increments | Tested |
| `ed6-it-et-overlap` | IT and ET neurons overlap electrophysiologically at 34 degrees | Tested |
| `ed6-it-et-ap-threshold` | AP threshold differs between L5 IT and ET at 34 degrees | Tested |
| `link-cells` | The files are the cells of the paper's published 34 degree table | Tested, with the paper's table |
| `link-features` | Features computed from the files agree with the published ones | Tested, with the paper's table |
| `ed6-sst-temperature` | Sst t-types keep their relative differences between 25 and 34 degrees | Tested, with the paper's room-temperature features |
| `ed6-sst-transcriptomic-distance` | Transcriptomic and electrophysiological distances of Sst t-types correlate at 34 degrees | Blocked: the reference expression data could not be fetched within the time limit |

The outside inputs are the paper's own tables in its code repository,
[berenslab/mini-atlas](https://github.com/berenslab/mini-atlas) at commit
`3bf98a524b8e48aa8647e1b9ae8f2351d9c712c9`. They supply only what the dandiset lacks: the
features of the room-temperature cells, and the paper's per-cell values to check the link. All
analysis of the 34 degree cells runs on the traces in the dandiset.

The features are computed by `scala34.py`, a re-implementation of the paper's feature
extraction (Methods, and the paper's code where the Methods leave a detail open).

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import scala34
from scala34 import FEATURES

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
COLORS = {"IT": "#c0392b", "ET": "#2471a3", "Sst": "#7d3c98", "Pvalb": "#1e8449", "NP": "#7f8c8d"}

## Load the data and check it against its description

`scala34.load_cohort` streams all 185 files in parallel and reduces each, as it loads, to one
row: the file's metadata, checks of the stimulus and timing, and the 17 features.

In [ ]:
cells = scala34.load_cohort()
F = cells[FEATURES]
print(f"dandiset {scala34.DANDISET}, version {scala34.VERSION}: {len(cells)} files, "
      f"{cells.cell_id.nunique()} distinct cell ids, {cells.subject_subject_id.nunique()} mice")
print("\nCells by mouse, genotype and family (family from the t-type name; NaN t-type = none):")
print(pd.crosstab([cells.subject_subject_id, cells.subject_genotype],
                  cells.family.fillna("no t-type"), margins=True).to_string())
print("\nInferred layer:", cells.inferred_layer.value_counts().to_dict(),
      "| targeted layer:", cells.targeted_layer.astype(str).value_counts().to_dict())
print("Recording temperature field:", cells.recording_temperature.value_counts().to_dict())
print("Session description:", cells.session_description.value_counts().to_dict())
print("Sampling rates (Hz):", sorted({r for rs in cells.rates for r in rs}),
      "| samples per response:", sorted({n for ns in cells.resp_samples for n in ns}),
      "| samples per stimulus:", sorted({n for ns in cells.stim_samples for n in ns}))
print("Stated units: voltage", sorted({u for us in cells.v_units for u in us}),
      "| stimulus", sorted({u for us in cells.i_units for u in us}))
print(f"Voltage values over all sweeps: {cells.v_min.min():.1f} to {cells.v_max.max():.1f} (stated unit volts)")
amps = np.concatenate(cells.amplitudes.to_numpy())
print(f"Stimulus values over all sweeps: {amps.min():.0f} to {amps.max():.0f} (stated unit amperes)")
print("Sweeps per cell: min", cells.n_sweeps.min(), "median", int(cells.n_sweeps.median()), "max", cells.n_sweeps.max())
print("Series with an undefined (NaN) starting time: all series of", cells.starting_times_nan.sum(), "cells")
print(f"Soma depth (um): {cells.soma_depth_um.min():.0f} to {cells.soma_depth_um.max():.0f}")
print(f"t-type confidence: {cells.RNA_type_confidence.min():.2f} to {cells.RNA_type_confidence.max():.2f}, "
      f"missing for {cells.RNA_type_confidence.isna().sum()} cells")
print("\nCells whose feature extraction raised an error:", (cells.feature_error != "").sum())
print("Missing values per feature:", F.isna().sum()[F.isna().sum() > 0].to_dict() or "none")
print("\nThe 17 features (min, median, max over cells):")
print(F.describe().T[["min", "50%", "max"]].round(2).to_string())

In [ ]:
# Dandiset-level metadata, and the draft against the pinned version.
from dandi.dandiapi import DandiAPIClient

client = DandiAPIClient()
meta = client.get_dandiset(scala34.DANDISET, scala34.VERSION).get_raw_metadata()
summary = meta["assetsSummary"]
print("name:", meta["name"])
print("description:", meta["description"].strip())
print("assetsSummary: subjects", summary.get("numberOfSubjects"), "| cells", summary.get("numberOfCells"),
      "| files", summary.get("numberOfFiles"), "| approach", summary.get("approach"),
      "| measurementTechnique", summary.get("measurementTechnique"),
      "| variableMeasured", summary.get("variableMeasured"))
print("about:", [(a.get("name"), a.get("schemaKey")) for a in meta.get("about", [])])
print("contributors named 'Last, First':",
      [(c["name"], c.get("identifier")) for c in meta["contributor"] if c["name"] == "Last, First"])
ids = pd.Series({c["name"]: c.get("identifier") for c in meta["contributor"]})
print("identifiers shared by more than one contributor:", ids[ids.notna() & ids.duplicated(keep=False)].to_dict())

published = {(a.path, a.size) for a in client.get_dandiset(scala34.DANDISET, scala34.VERSION).get_assets()}
draft = {(a.path, a.size) for a in client.get_dandiset(scala34.DANDISET, "draft").get_assets()}
print(f"\nassets: {len(published)} in {scala34.VERSION}, {len(draft)} in the draft, "
      f"{len(published ^ draft)} differ in path or size")
dmeta = client.get_dandiset(scala34.DANDISET, "draft").get_raw_metadata()
print("draft metadata: approach", [a["name"] for a in dmeta["assetsSummary"].get("approach", [])],
      "| measurementTechnique", [m["name"] for m in dmeta["assetsSummary"].get("measurementTechnique", [])],
      "| about", [(a.get("name"), a.get("schemaKey")) for a in dmeta.get("about", [])],
      "| contributors named 'Last, First':", sum(c["name"] == "Last, First" for c in dmeta["contributor"]))

In [ ]:
# Duplicate recordings: identical first sweeps, or identical sets of all sweeps.
print("cells sharing the first sweep with another cell:", cells.hash_first.duplicated(keep=False).sum())
print("cells sharing all sweeps with another cell:", cells.hash_all.duplicated(keep=False).sum())
print("cell ids used by more than one file:", cells.cell_id.duplicated(keep=False).sum())

In [ ]:
# Timing: does the voltage respond at the stimulus onset, in the direction of the current?
# For each cell: change of mean voltage from the 5 ms before to the 5 ms after the step onset
# (0.1 s in the stimulus series), against the same measure 50 ms earlier (no step there).
t = cells[["onset_step_hyper", "onset_step_hyper_null", "offset_step_hyper",
           "onset_step_depol", "onset_step_depol_null", "onset_latency_hyper_ms"]]
print(t.describe().T[["min", "50%", "max"]].round(3).to_string())
print("\ncells whose voltage falls at the onset of the most negative step:", (t.onset_step_hyper < 0).sum(), "of", len(t))
print("cells whose voltage rises at the onset of the first positive step:", (t.onset_step_depol > 0).sum(), "of", len(t))
print("cells with |change| > 1 mV 50 ms before the onset (null):",
      ((t.onset_step_hyper_null.abs() > 1) | (t.onset_step_depol_null.abs() > 1)).sum())
print("first positive step used for the check (pA):", cells.amp_depol_check.value_counts().to_dict())
print("\nLatency of the hyperpolarizing response (1 mV below baseline) from the step onset, ms:")
print(cells.groupby("subject_subject_id").onset_latency_hyper_ms.agg(["min", "median", "max"]).round(2).to_string())

Checks of the description against the data (the numbers are printed above):

- Units. Voltage series state the unit "volts" with a conversion of 1, but their values run
  from -150.3 to 66.5, and the median resting potential is -61.7: the values are millivolts.
  Stimulus series state "amperes" with a conversion of 1, but their values are -200 to 1,580:
  they are picoamperes. The analysis treats them as mV and pA.
- Timing. The stimulus series are 22,500 samples (0.9 s) and the responses 25,000 samples
  (1.0 s) at 25 kHz; every step starts at 0.1 s and ends at 0.7 s of its stimulus series. In
  every cell the voltage falls at the onset of the most negative step and rises at the onset of
  the first positive one, with no comparable change 50 ms earlier, and the hyperpolarizing
  response starts between -0.08 and 0.84 ms from the stimulus onset. So within a sweep the response and
  stimulus share a clock, consistent with the step onset at 0.1 s. Every series has an undefined
  (NaN) starting time, so the timing of sweeps relative to each other (the 1.3 to 1.4 s
  intervals of the Methods) cannot be checked.
- The 0 pA sweep is stored as an IZeroClampSeries with no stimulus series, which is why each
  file has one more response than stimulus series.
- Temperature. Every file states 33-34 degrees; the paper says 34 degrees.
- Duplicates. No two files share a cell id, a first sweep or a set of sweeps.
- Dandiset metadata. See the cell above: the description repeats the main data set's "over
  1300 neurons", the assets summary gives no technique or approach, one contributor is the
  placeholder "Last, First" with another author's identifier, and the brain region is typed as a
  disorder. The draft, which is not published, corrects these and has the same assets as the
  pinned version.

## Claim `ed6-cohort`

"We sequenced in total 185 neurons, obtained from 8 adult mice (7 Sst-Cre/Ai9 and 1
Pvalb-Cre/Ai9), of which 177 neurons passed the transcriptomic quality control and got a t-type
assignment. One hundred and ten cells mapped to the Sst subclass, 43 to IT, 12 to ET, 10 to
Pvalb, and 2 to NP. 175 cells were assigned to L5 ... We obtained high-quality
electrophysiological recordings and extracted electrophysiological features of 184 cells."
(Methods)

In [ ]:
genotype = cells.groupby("subject_subject_id").subject_genotype.first()
fam = cells.family.value_counts()
cohort = pd.DataFrame({
    "paper": [185, 8, 7, 1, 177, 110, 43, 12, 10, 2, 175, 184],
    "tolerance": [0] * 11 + [18.4],
    "here": [len(cells), cells.subject_subject_id.nunique(),
             genotype.str.startswith("Sst").sum(), genotype.str.startswith("Pvalb").sum(),
             cells.family.notna().sum(), fam.get("Sst", 0), fam.get("IT", 0), fam.get("ET", 0),
             fam.get("Pvalb", 0), fam.get("NP", 0), (cells.inferred_layer == "5").sum(),
             F.notna().all(axis=1).sum()]},
    index=["neurons", "mice", "Sst-Cre mice", "Pvalb-Cre mice", "neurons with a t-type", "Sst", "IT",
           "ET", "Pvalb", "NP", "layer 5", "neurons with all 17 features"])
cohort["within tolerance"] = (cohort.here - cohort.paper).abs() <= cohort.tolerance
print(cohort.to_string())
print("\nET t-types as named in the files:", sorted(cells.RNA_type[cells.family == "ET"].unique()))
print("Cells without a t-type:", cells.cell_id[cells.family.isna()].tolist())

**Result: reproduced.** The files hold 185 neurons from 8 mice, 7 of them Sst-Cre and 1
Pvalb-Cre; 177 have a t-type (110 Sst, 43 IT, 12 ET, 10 Pvalb, 2 NP) and 175 are in layer 5,
all as the Methods state. All 17 features could be computed for all 185 neurons, against 184 in
the paper, within the 10% tolerance. The 12 ET neurons carry the older names L5 PT_1 to
L5 PT_4 in the files.

## Claim `ed6-stimulus-protocol`

"Each neuron was injected with 600-ms-long current pulses starting from -200 pA and up to
1,380 pA with 20-pA increment steps (in some cases stimulation was stopped before reaching
1,380 pA)." The 34 degree experiments used "the same stimulation paradigm". (Methods)

In [ ]:
def protocol(row):
    a = np.array(sorted(row.amplitudes))
    on, off = row.step_onsets, row.step_offsets
    return pd.Series({
        "duration ok": abs((off[0] - on[0]) - 0.6) <= 0.001 and abs((off[1] - on[1]) - 0.6) <= 0.001,
        "first step (pA)": a[0], "steps of 20 pA": bool(np.all(np.round(np.diff(a)) == 20)),
        "last step (pA)": a[-1], "sweeps": len(a)})

proto = cells.apply(protocol, axis=1)
proto["first is -200"] = proto["first step (pA)"] == -200
proto["last <= 1380"] = proto["last step (pA)"] <= 1380
proto["conforms"] = proto[["duration ok", "steps of 20 pA", "first is -200", "last <= 1380"]].all(axis=1)
print(proto[["duration ok", "steps of 20 pA", "first is -200", "last <= 1380", "conforms"]].sum().to_string())
print(f"\n{proto.conforms.sum()} of {len(proto)} cells conform ({100 * proto.conforms.mean():.1f}%)")
print("\nCells that do not conform:")
print(pd.concat([cells[["cell_id", "subject_subject_id"]], proto], axis=1)[~proto.conforms].to_string())
print("\nLast step over all cells (pA):", proto["last step (pA)"].value_counts().sort_index().to_dict())

**Result: partially reproduced.** In all 185 cells every step lasts 600 ms and consecutive
steps differ by 20 pA, and 139 cells end at 1,380 pA. 174 of 185 cells (94.1%) conform to the
described ladder. Nine cells of mouse_UEQOU end above 1,380 pA (1,400 to 1,580 pA), and two
cells of other mice start at -180 pA and -100 pA and also end above 1,380 pA.

## Claim `ed6-it-et-overlap`

"We confirmed the electrophysiological overlap between IT and ET neurons in follow-up
experiments at 34 degrees C (Extended Data Fig. 6)." At room temperature the overlap is the
kNN confusion of Fig. 3d, in which "only the ET and IT subclasses [were] strongly overlapping".

The paper's classifier is applied as its code does it (see the criterion in the claims file):
k = 10, leave-one-out, on the paper's 17-feature representation of all cells with all features.

In [ ]:
rng_overlap = np.random.default_rng(3501)
ok = F.notna().all(axis=1).to_numpy()
X_all = scala34.representation(cells[ok])
fam_all = cells.family[ok].to_numpy()
classes = [c for c in ["Lamp5", "Vip", "Sst", "Pvalb", "IT", "ET", "CT"] if (fam_all == c).any()]
C, counted = scala34.knn_family_confusion(X_all, fam_all, classes, k=10)
print(f"{ok.sum()} cells in the representation; classes scored: {classes}")
print("cells per class that were counted (majority family among the classes):", counted.to_dict())
print("\nConfusion matrix, % of each row (true family) classified as each column:")
print((100 * C).round(1).to_string())
et_as_it, it_as_et = C.loc["ET", "IT"], C.loc["IT", "ET"]
print(f"\nET classified as IT: {100 * et_as_it:.1f}%   IT classified as ET: {100 * it_as_et:.1f}%")
print("Chance reference: share of IT among IT and ET cells "
      f"{(fam_all == 'IT').sum() / np.isin(fam_all, ['IT', 'ET']).sum():.2f}")

In [ ]:
# By mouse: IT and ET cells come from three mice.
from sklearn.neighbors import NearestNeighbors
idx = NearestNeighbors(n_neighbors=10).fit(X_all).kneighbors(return_distance=False)
labels = np.array([f if isinstance(f, str) else "" for f in fam_all])
pred = []
for i in range(len(labels)):
    u, c = np.unique(labels[idx[i]], return_counts=True)
    pred.append(u[np.argmax(c)])
pred = np.array(pred)
mice = cells.subject_subject_id[ok].to_numpy()
exc = np.isin(labels, ["IT", "ET"])
print(pd.crosstab([mice[exc], labels[exc]], pred[exc], rownames=["mouse", "true"], colnames=["kNN"]).to_string())

In [ ]:
# Variants (do not decide the outcome).
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import balanced_accuracy_score

# (a) Layer 5 cells only, representation recomputed on them.
l5 = (cells.inferred_layer == "5").to_numpy() & ok
X5 = scala34.representation(cells[l5])
fam5 = cells.family[l5].to_numpy()
C5, n5 = scala34.knn_family_confusion(X5, fam5, [c for c in classes if (fam5 == c).any()], k=10)
print(f"(a) layer 5 only ({l5.sum()} cells): ET as IT {100 * C5.loc['ET', 'IT']:.1f}%, "
      f"IT as ET {100 * C5.loc['IT', 'ET']:.1f}%, counted {n5.to_dict()}")

# (b) Balanced: IT subsampled to the number of ET cells, kNN among IT and ET cells only.
it_i, et_i = np.flatnonzero(fam_all == "IT"), np.flatnonzero(fam_all == "ET")
k_bal = min(10, 2 * len(et_i) - 1)
res = []
for _ in range(200):
    pick = np.concatenate([rng_overlap.choice(it_i, len(et_i), replace=False), et_i])
    Xb, fb = X_all[pick], fam_all[pick]
    nb = NearestNeighbors(n_neighbors=k_bal).fit(Xb).kneighbors(return_distance=False)
    guess = np.array([("IT" if (fb[n] == "IT").sum() > k_bal / 2 else "ET") for n in nb])
    res.append(((guess[fb == "ET"] == "IT").mean(), (guess[fb == "IT"] == "ET").mean()))
res = np.array(res)
print(f"(b) balanced, k = {k_bal}, {len(et_i)} + {len(et_i)} cells, 200 subsamples: ET as IT "
      f"{100 * res[:, 0].mean():.1f}% (5-95%: {100 * np.percentile(res[:, 0], 5):.0f}-{100 * np.percentile(res[:, 0], 95):.0f}), "
      f"IT as ET {100 * res[:, 1].mean():.1f}% (5-95%: {100 * np.percentile(res[:, 1], 5):.0f}-{100 * np.percentile(res[:, 1], 95):.0f}); "
      "50% each if inseparable")

# (c) Logistic regression IT against ET, layer 5, class-balanced, repeated stratified 5-fold CV.
sel = np.isin(fam5, ["IT", "ET"])
Xc, yc = X5[sel], (fam5[sel] == "ET").astype(int)
def cv_bacc(y, seed):
    p = cross_val_predict(LogisticRegression(class_weight="balanced", max_iter=1000), Xc, y,
                          cv=StratifiedKFold(5, shuffle=True, random_state=seed))
    return balanced_accuracy_score(y, p)
real = np.array([cv_bacc(yc, s) for s in range(20)])
null = np.array([cv_bacc(rng_overlap.permutation(yc), s) for s in range(200)])
print(f"(c) logistic regression, layer 5 IT ({(yc == 0).sum()}) against ET ({(yc == 1).sum()}): balanced accuracy "
      f"{real.mean():.2f} (range over 20 CV splits {real.min():.2f}-{real.max():.2f}); "
      f"label-shuffled null mean {null.mean():.2f}, 95th percentile {np.percentile(null, 95):.2f}, "
      f"p = {(1 + (null >= real.mean()).sum()) / (1 + len(null)):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for f_ in ["Sst", "Pvalb", "NP", "IT", "ET"]:
    m = fam_all == f_
    axes[0].scatter(X_all[m, 0], X_all[m, 1], s=12, color=COLORS[f_], label=f"{f_} ({m.sum()})")
axes[0].set_xlabel("PC1 (scaled)"); axes[0].set_ylabel("PC2 (scaled)"); axes[0].legend(frameon=False, fontsize=8)
axes[0].set_title("All cells, paper's representation")
l5c = cells[(cells.inferred_layer == "5") & cells.family.isin(["IT", "ET"])]
for f_ in ["IT", "ET"]:
    m = l5c.family == f_
    axes[1].scatter(l5c.loc[m, "Input resistance (MOhm)"], l5c.loc[m, "AP threshold (mV)"], s=14,
                    color=COLORS[f_], label=f"L5 {f_} ({m.sum()})")
axes[1].set_xlabel("Input resistance (MOhm)"); axes[1].set_ylabel("AP threshold (mV)"); axes[1].legend(frameon=False)
axes[1].set_title("As Extended Data Fig. 6f")
plt.tight_layout()
print(f"PC1 {X_all[:, 0].min():.1f} to {X_all[:, 0].max():.1f}, PC2 {X_all[:, 1].min():.1f} to {X_all[:, 1].max():.1f}; "
      f"input resistance {l5c['Input resistance (MOhm)'].min():.0f} to {l5c['Input resistance (MOhm)'].max():.0f} MOhm, "
      f"AP threshold {l5c['AP threshold (mV)'].min():.1f} to {l5c['AP threshold (mV)'].max():.1f} mV; all points shown")

**Result: reproduced.** With the paper's classifier, 58.3% of ET cells (7 of 12) are classified
as IT, above the 25% the criterion takes as overlap; no IT cell is classified as ET, and all Sst
cells are classified as Sst. ET cells go to IT in both mice that have them (1 of 3 in
mouse_MITSU, 6 of 9 in mouse_UEQOU). The overlap is one-sided and partly reflects the larger
number of IT cells (78% of IT and ET cells). With IT subsampled to 12 cells, 4.1% of ET cells
go to IT and 47.5% of IT cells to ET, and a logistic regression tells layer 5 IT from ET with a
balanced accuracy of 0.83, against 0.50 for shuffled labels (p = 0.005). The two subclasses
overlap in the nearest-neighbour sense the paper uses, but a linear classifier separates them
well above chance.

## Claim `ed6-it-et-ap-threshold`

Extended Data Fig. 6a-e compares layer 5 IT and ET neurons at both temperatures on the features
that differed most at room temperature, and "the last panel additionally shows one feature that
showed prominent difference at 34 degrees C", which the paper's code shows is the AP threshold.

In [ ]:
l5 = cells[(cells.inferred_layer == "5")]
it, et = l5[l5.family == "IT"], l5[l5.family == "ET"]
rows = []
for name in FEATURES:
    a, b = it[name].dropna(), et[name].dropna()
    u = stats.mannwhitneyu(a, b, alternative="two-sided")
    rows.append({"feature": name, "median IT": a.median(), "median ET": b.median(),
                 "AUC (P[IT > ET])": u.statistic / (len(a) * len(b)), "p": u.pvalue})
mw = pd.DataFrame(rows).sort_values("p").reset_index(drop=True)
mw.index += 1
print(f"Layer 5 IT: {len(it)} cells, layer 5 ET: {len(et)} cells")
print(mw.round(4).to_string())
rank = mw.index[mw.feature == "AP threshold (mV)"][0]
print(f"\nAP threshold: rank {rank} of {len(mw)}, p = {mw.loc[rank, 'p']:.2g}")
print("Features with the largest differences at room temperature in the paper's code (sag ratio, input "
      "resistance, AP width, rebound) rank here:",
      {f: int(mw.index[mw.feature == f][0]) for f in ["Sag ratio", "Input resistance (MOhm)", "AP width (ms)", "Rebound (mV)"]})
print("\nAP threshold by mouse (median mV, cells):")
print(l5[l5.family.isin(["IT", "ET"])].groupby(["subject_subject_id", "family"])["AP threshold (mV)"]
      .agg(["median", "size"]).round(1).to_string())

**Result: reproduced.** At 34 degrees the AP threshold of layer 5 IT cells (median -27.0 mV,
43 cells) is higher than that of layer 5 ET cells (-33.1 mV, 12 cells), Mann-Whitney
p = 0.00035, the second smallest p of the 17 features after input resistance. The difference
has the same sign in both mice with IT and ET cells (mouse_MITSU -25.6 against -37.1 mV,
mouse_UEQOU -29.3 against -31.8 mV). Of the four features that differ most at room temperature
in the paper's code, input resistance ranks first here, AP width sixth, sag ratio ninth
(p = 0.052) and rebound fifteenth.

## Claim `link-cells` (with the paper's published table)

The paper's repository publishes a per-cell table of the 34 degree cohort
(`m1_patchseq_phys_temp_meta_data.csv`). Are the dandiset's files those cells, with the same
labels?

In [ ]:
pub_meta = scala34.published_table("m1_patchseq_phys_temp_meta_data.csv", sep="\t")
print(f"published table: {len(pub_meta)} rows, columns include {list(pub_meta.columns[:6])} ...")
link = cells.merge(pub_meta, left_on="cell_id", right_on="Cell", how="left", indicator=True)
matched = link[link._merge == "both"].copy()
print(f"dandiset cells matched by cell id: {len(matched)} of {len(cells)}; "
      f"table rows without a file: {len(set(pub_meta.Cell) - set(cells.cell_id))}")

def same(a, b):
    return (a.astype(str) == b.astype(str)) | (a.isna() & b.isna())

matched["layer_from_depth"] = scala34.layer_from_depth(matched["Soma depth (µm)"].astype(float),
                                                       matched["Cortical thickness (µm)"].astype(float))
agree = pd.Series({
    "t-type": same(matched.RNA_type, matched["RNA type"]).mean(),
    "family (from name) vs table's family": same(matched.family.fillna("low quality"), matched["RNA family"]).mean(),
    "layer: file's inferred layer vs layer from depth and thickness": same(matched.inferred_layer, matched.layer_from_depth).mean(),
    "layer: file's inferred layer vs table's inferred layer": same(matched.inferred_layer, matched["Inferred layer"]).mean(),
    "soma depth": (matched.soma_depth_um.astype(float) == matched["Soma depth (µm)"].astype(float)).mean(),
    "mouse": same(matched.subject_subject_id, matched.Mouse).mean(),
})
print("\nFraction of matched cells that agree:")
print(agree.round(3).to_string())
tt = ~same(matched.RNA_type, matched["RNA type"])
print("\nt-types that differ, file name against table name (cells):")
print(matched[tt].groupby(["RNA_type", "RNA type"]).size().to_string())
renamed = matched.RNA_type.str.replace(" PT_", " ET_", regex=False)
print(f"agreement on t-type after renaming L5 PT_k to L5 ET_k: {same(renamed, matched['RNA type']).mean():.3f}")
print("\nLayer from depth and thickness:", pd.Series(matched.layer_from_depth).value_counts().to_dict())
print("Cells that disagree on layer:")
print(matched.loc[~same(matched.inferred_layer, matched.layer_from_depth),
                  ["cell_id", "inferred_layer", "layer_from_depth", "Soma depth (µm)", "Cortical thickness (µm)"]].to_string())
print("Information in the table but not in the files: cortical thickness, recording temperature as a number "
      "and the family column")

**Result: partially reproduced.** All 185 files match a row of the paper's table by cell id,
and every matched cell agrees on family, layer (the file's inferred layer equals the layer the
paper's code computes from soma depth and cortical thickness), soma depth and mouse. The t-type
name agrees for 93.5% of cells, below the criterion's 95%: the 12 ET cells are named L5 PT_1 to
L5 PT_4 in the files and L5 ET_1 to L5 ET_4 in the table, one to one, and with that renaming
the agreement is 100%. The files do not hold the cortical thickness the paper uses to assign
layers, but their inferred layer equals the paper's assignment.

## Claim `link-features` (with the paper's published features)

Do the features computed here from the dandiset's traces agree with the paper's published
features for the same cells (`m1_patchseq_phys_temp_ephys_features.csv`)?

In [ ]:
pub_feat = scala34.published_table("m1_patchseq_phys_temp_ephys_features.csv").set_index("cell id")
print(f"published features: {len(pub_feat)} cells; dandiset cells not in it:",
      sorted(set(cells.cell_id) - set(pub_feat.index)))
both = cells.set_index("cell_id").join(pub_feat, how="inner", rsuffix=" (paper)")
rows = []
for name in FEATURES:
    a, b = both[name], both[name + " (paper)"]
    m = a.notna() & b.notna()
    rows.append({"feature": name, "cells": m.sum(), "Spearman": stats.spearmanr(a[m], b[m]).statistic,
                 "median here": a[m].median(), "median paper": b[m].median(),
                 "median |difference|": (a[m] - b[m]).abs().median(),
                 "cells within 10% of paper": ((a[m] - b[m]).abs() <= 0.1 * b[m].abs()).mean()})
agreement = pd.DataFrame(rows).set_index("feature")
print(agreement.round(3).to_string())
print(f"\nfeatures with Spearman >= 0.9: {(agreement.Spearman >= 0.9).sum()} of 17; "
      f">= 0.8: {(agreement.Spearman >= 0.8).sum()} of 17; lowest {agreement.Spearman.min():.3f}")
code_max = both["Max number of APs (code)"]
within = pd.DataFrame({name: (both[name] - both[name + " (paper)"]).abs() <= 0.1 * both[name + " (paper)"].abs()
                       for name in FEATURES})
print("\nBy mouse: cells, and the fraction of (cell, feature) values within 10% of the paper's")
print(pd.DataFrame({"cells": both.groupby("subject_subject_id").size(),
                    "within 10%": within.groupby(both.subject_subject_id).mean().mean(axis=1).round(3)}).to_string())
print(f"\nMax number of APs, the code's reading (before excluding sweeps with spikes outside the step): "
      f"differs from the Methods' reading in {(code_max != both['Max number of APs']).sum()} cells; "
      f"Spearman with the paper {stats.spearmanr(code_max, both['Max number of APs (paper)']).statistic:.3f}")

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(15, 7.5))
for ax, name in zip(axes.ravel(), FEATURES):
    a, b = both[name], both[name + " (paper)"]
    ax.scatter(b, a, s=5, color="k")
    lo, hi = np.nanmin([a.min(), b.min()]), np.nanmax([a.max(), b.max()])
    ax.plot([lo, hi], [lo, hi], color="0.7", lw=0.8)
    ax.set_title(f"{name}\nrho = {agreement.loc[name, 'Spearman']:.2f}", fontsize=8)
    ax.tick_params(labelsize=7)
axes.ravel()[-1].axis("off")
fig.supxlabel("published value"); fig.supylabel("computed here")
plt.tight_layout()
print("All matched cells shown; each axis spans the full range of both values.")

**Result: reproduced.** Of the 185 cells, 184 are in the paper's feature table
(20200406_sample_4 is not), as in the paper's count of 184. For 16 of the 17 features the
Spearman correlation between the values computed here and the published values is at least
0.9; the lowest is the AP coefficient of variation, 0.877, above the criterion's 0.8. The median
absolute difference is zero to three decimals for every feature but the AHP (0.011 mV). By
mouse, 0.895 (mouse_UEQOU) to 0.995 of values are within 10% of the paper's. So the traces in
the dandiset yield the paper's published features. Taking the maximum number of APs as the
paper's code does, before excluding sweeps with spikes outside the step, changes it for 3 cells.

## Claim `ed6-sst-temperature` (with the paper's room-temperature features)

"The change of electrophysiological properties between room temperature (25 degrees C) and
physiological temperature (34 degrees C) for various t-types from the Sst subclass. Only L5
neurons are shown. Only t-types with >= 5 cells in both conditions are shown ... AP amplitude
and AP width changed the most between conditions, but the relative differences between t-types
stayed roughly the same. The other four shown features did not change much, and the relative
differences between t-types stayed the same." (Extended Data Fig. 6h)

Room-temperature values are the paper's published features of its main data set; 34 degree
values are computed here from the dandiset.

In [ ]:
SHOWN = ["AP amplitude (mV)", "AP width (ms)", "Afterhyperpolarization (mV)",
         "Input resistance (MOhm)", "Rebound (mV)", "Sag ratio"]
main_meta = scala34.published_table("m1_patchseq_meta_data.csv", sep="\t")
main_feat = scala34.published_table("m1_patchseq_ephys_features.csv").set_index("cell id")
depth = pd.to_numeric(main_meta["Soma depth (µm)"], errors="coerce")
thick = pd.to_numeric(main_meta["Cortical thickness (µm)"], errors="coerce").replace(0, np.nan)
layer = main_meta["Targeted layer"].astype(str).to_numpy().astype(object)
from_depth = scala34.layer_from_depth(depth, thick)
layer[pd.notna(from_depth)] = from_depth[pd.notna(from_depth)]
main_meta["layer"] = layer
excluded = main_meta["Exclusion reasons"].fillna("").astype(str) != ""
rt = main_meta[(main_meta["RNA family"] == "Sst") & (main_meta.layer == "5") & ~excluded]
rt = rt.join(main_feat, on="Cell", how="inner")
print(f"room temperature: {len(main_meta)} cells in the table, {excluded.sum()} excluded; "
      f"{len(rt)} layer 5 Sst cells with features")
hot = cells[(cells.family == "Sst") & (cells.inferred_layer == "5")]
print(f"34 degrees (dandiset): {len(hot)} layer 5 Sst cells")
n_rt, n_hot = rt["RNA type"].value_counts(), hot.RNA_type.value_counts()
types = sorted(t for t in set(n_rt.index) & set(n_hot.index) if n_rt[t] >= 5 and n_hot[t] >= 5)
print(pd.DataFrame({"25 C": n_rt, "34 C": n_hot}).fillna(0).astype(int).loc[
    sorted(set(n_rt.index) | set(n_hot.index))].to_string())
print(f"\nt-types with at least 5 cells in both: {len(types)}: {types}")

In [ ]:
rows = []
for name in SHOWN:
    m25 = rt[rt["RNA type"].isin(types)].groupby("RNA type")[name].median().loc[types]
    m34 = hot[hot.RNA_type.isin(types)].groupby("RNA_type")[name].median().loc[types]
    sd = rt[name].std()
    rows.append({"feature": name, "median 25 C": rt[name].median(), "median 34 C": hot[name].median(),
                 "shift (SD at 25 C)": np.median(m34.to_numpy() - m25.to_numpy()) / sd,
                 "Spearman across t-types": stats.spearmanr(m25, m34).statistic,
                 "Pearson across t-types": stats.pearsonr(m25, m34).statistic})
temp = pd.DataFrame(rows).set_index("feature")
temp["|shift| rank"] = temp["shift (SD at 25 C)"].abs().rank(ascending=False).astype(int)
print(temp.round(3).to_string())
top2 = set(temp["|shift| rank"].nsmallest(2).index)
part1 = top2 == {"AP amplitude (mV)", "AP width (ms)"}
rho = temp["Spearman across t-types"]
part2 = bool((rho > 0).all() and rho.median() >= 0.5)
print(f"\n(i) two largest shifts: {sorted(top2)} -> {'holds' if part1 else 'does not hold'}")
print(f"(ii) Spearman positive for all six: {bool((rho > 0).all())}; median {rho.median():.2f} -> "
      f"{'holds' if part2 else 'does not hold'}")

In [ ]:
# Variants: the same with the paper's published 34 degree features in place of those computed
# here (separates the extraction from the data), and with the t-type means in place of medians.
pub34 = hot[["cell_id", "RNA_type"]].join(pub_feat, on="cell_id", how="inner")
var = []
for name in SHOWN:
    m25 = rt[rt["RNA type"].isin(types)].groupby("RNA type")[name].median().loc[types]
    p34 = pub34[pub34.RNA_type.isin(types)].groupby("RNA_type")[name].median().loc[types]
    a25 = rt[rt["RNA type"].isin(types)].groupby("RNA type")[name].mean().loc[types]
    a34 = hot[hot.RNA_type.isin(types)].groupby("RNA_type")[name].mean().loc[types]
    var.append({"feature": name,
                "published 34 C: shift (SD)": np.median(p34.to_numpy() - m25.to_numpy()) / rt[name].std(),
                "published 34 C: Spearman": stats.spearmanr(m25, p34).statistic,
                "means: shift (SD)": np.median(a34.to_numpy() - a25.to_numpy()) / rt[name].std(),
                "means: Spearman": stats.spearmanr(a25, a34).statistic})
print(pd.DataFrame(var).set_index("feature").round(3).to_string())

fig, axes = plt.subplots(1, 6, figsize=(16, 3.4))
for ax, name in zip(axes, SHOWN):
    for x, t_ in enumerate(types):
        a = rt.loc[rt["RNA type"] == t_, name]; b = hot.loc[hot.RNA_type == t_, name]
        ax.scatter(np.full(len(a), x - 0.15), a, s=4, color="0.6")
        ax.scatter(np.full(len(b), x + 0.15), b, s=4, color="k")
        ax.plot([x - 0.3, x], [a.median()] * 2, color="0.6"); ax.plot([x, x + 0.3], [b.median()] * 2, color="k")
    ax.set_xticks(range(len(types))); ax.set_xticklabels([t_.replace("Sst ", "") for t_ in types], rotation=70, fontsize=7)
    ax.set_title(name, fontsize=8)
fig.suptitle("Layer 5 Sst t-types: 25 C (grey, published) and 34 C (black, computed here); lines are medians", fontsize=9)
plt.tight_layout()
print("All cells of the shown t-types are plotted; ranges:",
      {n: (round(float(min(rt[n].min(), hot[n].min())), 2), round(float(max(rt[n].max(), hot[n].max())), 2)) for n in SHOWN})

In [ ]:
# By mouse at 34 degrees: the two features that shift most, for the layer 5 Sst cells of the shown
# t-types, against the room-temperature median of the same t-types (published, all mice pooled).
shown_rt = rt[rt["RNA type"].isin(types)]
shown_hot = hot[hot.RNA_type.isin(types)]
print("room temperature, all mice:", {n: round(float(shown_rt[n].median()), 2) for n in ["AP amplitude (mV)", "AP width (ms)"]})
print(shown_hot.groupby("subject_subject_id")[["AP amplitude (mV)", "AP width (ms)"]].median().round(2)
      .join(shown_hot.groupby("subject_subject_id").size().rename("cells")).to_string())

**Result: reproduced.** Seven layer 5 Sst t-types have at least 5 cells at both temperatures.
From 25 to 34 degrees, AP width (shift -1.78 SD) and AP amplitude (-1.66 SD) change most,
then input resistance (-0.61 SD); AHP, rebound and sag ratio shift by 0.33 SD or less. The
order of t-types is kept: the Spearman correlation of t-type medians between temperatures is
0.75 for AP amplitude and 0.88 to 0.96 for the other five, median 0.93. The same holds with the
paper's published 34 degree features in place of those computed here, and with t-type means.
In each of the 7 mice with such cells, the 34 degree medians of AP amplitude (63.9 to 73.8 mV) and AP
width (0.52 to 0.64 ms) lie below the room-temperature medians of the same t-types (83.0 mV,
0.92 ms).

## Summary

| Claim | Data | Outcome |
|---|---|---|
| `fig1-ttype-coverage` | main data set | Blocked: data missing (dandiset 000008 and sequencing archives) |
| `fig2-rrr-prediction` | main data set | Blocked: data missing |
| `ed5-rrr-ion-channels` | main data set | Blocked: data missing |
| `fig3d-families-separate` | main data set | Blocked: data missing |
| `fig3d-morphology-combined` | main data set | Blocked: data missing |
| `fig4c-distance-correlation` | main data set | Blocked: data missing |
| `fig4d-it-depth` | main data set | Blocked: data missing |
| `fig5a-ttypes-between-families` | main data set | Blocked: data missing |
| `fig5b-ttype-variability` | main data set | Blocked: data missing |
| `ed6-cohort` | dandiset | Reproduced |
| `ed6-stimulus-protocol` | dandiset | Partially reproduced: 174 of 185 cells follow the stated ladder |
| `ed6-it-et-overlap` | dandiset | Reproduced |
| `ed6-it-et-ap-threshold` | dandiset | Reproduced |
| `link-cells` | with the paper's table | Partially reproduced: ET t-types carry older names in the files |
| `link-features` | with the paper's table | Reproduced |
| `ed6-sst-temperature` | with the paper's room-temperature features | Reproduced |
| `ed6-sst-transcriptomic-distance` | with reference expression data | Blocked: the reference data could not be fetched within the time limit |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: The features are computed by a
re-implementation of the paper's extraction (`scala34.py`), not by the paper's code; it gives
the published values for most cells. The maximum number of APs follows the Methods, which
exclude sweeps with spikes outside the step before taking the maximum; the paper's code does
not. Families are read from the t-type names in the files. Room-temperature values for
`ed6-sst-temperature` are the paper's published features, not recomputed from dandiset 000008.
The variants of `ed6-it-et-overlap` (balanced classes, logistic regression) are additions.

What was found about the dandiset:

- It holds only the follow-up cohort at physiological temperature; the description repeats the
  main data set's "over 1300 neurons" and points to dandiset 000008 for it.
- Voltage is stored in millivolts and current in picoamperes, while the files state volts and
  amperes with a conversion of 1.
- Every series has an undefined (NaN) starting time, so the timing between sweeps is lost.
- The 12 ET cells are labelled with the older t-type names L5 PT_1 to L5 PT_4, which differ from
  the names in the paper and its tables.
- The files hold no cortical thickness, which the paper uses to assign layers, but the stored
  inferred layer matches the paper's assignment for every cell.
- Eleven cells were stimulated beyond the 1,380 pA the Methods give, and two of them did not start
  at -200 pA.
- Dandiset metadata: no technique or approach in the assets summary, a placeholder contributor
  "Last, First" with another author's ORCID, and the brain region typed as a disorder in the
  published version. The unpublished draft corrects these.
- Everything else checked agrees with the paper: cell, mouse, t-type and layer counts, the
  600 ms step timing, and the published per-cell features.